# מעבדה 07 — אופני תנודה עצמיים ושרשרת N המסות

מודול 06 השאיר אתכם מחפשים ביד יחסי התנעה מיוחדים. זה עובד עבור שתי מסות ומפסיק לעבוד מיד
אחר כך. כאן אתם בונים את המכונה שעושה זאת עבור כל מספר מסות שהוא, משתמשים בה כדי לצפות בשרשרת
שפרטו עליה מסרבת לחלוק את האנרגיה שלה בין האופנים, ואז מודדים שרשרת מסות שהופכת למיתר.

שני החלקים האחרונים הם מדידות עם סרגלי שגיאה. אחד מהם יפתיע אתכם: התשובה כמעט אינה זזה כשאתם
מעלים את הרעש פי חמישה-עשר, מפני שמה שמגביל אותה אינו הרעש.

עבדו לפי הסדר. היכן שהמחברת מבקשת מכם לנבא, רשמו את הניבוי שלכם בתא המיועד **לפני** שאתם
מריצים את התא הבא.

## מפרט המודל

| | |
|---|---|
| **המערכת** | $N$ מסות שוות $m$ בטור על קפיצים זהים $k_s$ במרווח $a$, קצוות קבועים אלא אם נאמר אחרת |
| **הדינמיקה** | $M\ddot{x} + Kx = 0$, נפתרת בדיוק בפירוק לאופנים ומקודמת בנפרד בשיטת ורלה-מהירות |
| **תנאי השפה** | קירות קשיחים; הווריאנטים החופשי והמחזורי נושאים אופן בתדירות אפס, וזו פיזיקה |
| **האנסמבל** | מסלול דטרמיניסטי אחד לכל קבוצת תנאי התחלה; רעש המדידה גאוסי, בלתי תלוי ובעל זרע קבוע |
| **מה מוזנח** | ריסון, הנעה, מסת הקפיצים, אי-ליניאריות, היענות הקירות, ההבחנה בין רוחבי לאורכי |
| **תקף כאשר** | הקפיצים ליניאריים, ההזזות קטנות, וטענות רצף נטענות רק עבור $p \ll N$ |
| **אופני כשל** | מטריצות לא סימטריות; נוסחאות רצף ליד קצה הסרט; משרעות שבהן אי-ליניאריות מצמדת את האופנים |

כל הפיזיקה נמצאת ב-`wavelab.coupled` — פתחו וקראו.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import coupled, fourier, measurement
from wavelab.validation import convergence_study, seed_study

MASS = 0.5  # kg
STIFFNESS = 8.0  # N/m, the spring between neighbours — k_s on the page, since k is a wavenumber
LENGTH = 1.0  # m, wall to wall: the spacing is LENGTH/(N + 1) whatever N is

CEILING = 2 * np.sqrt(STIFFNESS / MASS)  # the fastest a chain of these parts can ever go

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"band top 2 sqrt(k_s/m) = {CEILING} rad/s — no chain of these parts exceeds this")
for n in (2, 5, 20, 100):
    omega = coupled.chain_mode_frequencies(n, MASS, STIFFNESS)
    print(f"N = {n:3d}: {n} modes, from {omega[0]:.4f} to {omega[-1]:.4f} rad/s")

## חלק 1 — המכונה הכללית, נבדקת מול משהו שאתם כבר יודעים

מודול 06 פתר את זוג שתי המסות בדטרמיננטה וקיבל $\omega_s = \sqrt{k/m}$ עם צורה $(1,1)$,
ו-$\omega_a = \sqrt{(k + 2k_c)/m}$ עם צורה $(1,-1)$. `normal_mode_solve` אינו יודע דבר מכל
זה — הוא בונה את $L^{-1}KL^{-\mathsf T}$ ומוסר אותה לפותר ערכים עצמיים סימטרי.

אם המכונה נכונה, היא חייבת לשחזר את תשובת היד במדויק. בדקו זאת לפני שאתם סומכים עליה על עשרים
מסות, שם אין לכם דבר לבדוק מולו.

In [ ]:
COUPLING = 0.05 * STIFFNESS
pair = coupled.two_mass_matrices(MASS, STIFFNESS, COUPLING)
pair_modes = coupled.normal_mode_solve(*pair)

by_hand = np.array([
    np.sqrt(STIFFNESS / MASS),
    np.sqrt((STIFFNESS + 2 * COUPLING) / MASS),
])
print(f"by determinant : {by_hand}")
print(f"by eigensolver : {pair_modes.frequencies}")
print(f"agreement      : {np.max(np.abs(pair_modes.frequencies - by_hand)):.2e} rad/s")
print()
print("shapes (columns), scaled so the largest entry is 1:")
print(np.round(pair_modes.shapes / np.abs(pair_modes.shapes).max(axis=0), 6))
print()
print("M-orthonormality, shapes^T M shapes — should be the identity:")
print(np.round(pair_modes.shapes.T @ pair[0] @ pair_modes.shapes, 12))

## חלק 2 — לנבא

לפני שאתם מריצים את התא הבא, התחייבו לתשובות.

1. שרשרת בת חמש מסות בין קירות קבועים. שרטטו את צורת האופן האיטי ביותר שלה ואת צורת המהיר
   ביותר. בכמה מקומות בכל אחת מהן מסה אינה זזה לעולם?
2. מה גדול יותר, התדירות המהירה ביותר של שרשרת בת 5 או של שרשרת בת 100 — ובאיזה יחס בערך?

**הניבוי שלכם:**

1.
2.

## חלק 3 — גלריית האופנים

חמש מסות, חמישה אופנים. הצורות יוצאות מפותר הערכים העצמיים; עקומות הסינוס המשורטטות דרכן הן
$\sin(p\pi x/L)$ בלי שום התאמה. שימו לב שלפותר נמסרה מטריצה ואיש לא אמר לו מהו סינוס.

In [ ]:
N_GALLERY = 5
matrices = coupled.chain_matrices(N_GALLERY, MASS, STIFFNESS)
modes = coupled.normal_mode_solve(*matrices)
exact = coupled.chain_mode_frequencies(N_GALLERY, MASS, STIFFNESS)

sites = np.arange(1, N_GALLERY + 1) * LENGTH / (N_GALLERY + 1)
smooth = np.linspace(0, LENGTH, 400)

fig, axes = plt.subplots(1, N_GALLERY, figsize=(12, 2.4), sharey=True)
for p, axis in enumerate(axes):
    shape = modes.shapes[:, p] / np.abs(modes.shapes[:, p]).max()
    # Match the drawn sine to the solved shape's sign: eigenvectors carry no preferred one.
    # The solver fixes each shape's sign by its own convention, which need not be the
    # sine's. Match them at the largest entry so the comparison is of shapes, not signs.
    peak = int(np.argmax(np.abs(shape)))
    sign = np.sign(shape[peak] * np.sin((p + 1) * np.pi * sites[peak] / LENGTH))
    curve = sign * np.sin((p + 1) * np.pi * smooth / LENGTH)
    axis.plot(smooth, curve, lw=0.9, ls="--", color="0.6")
    axis.plot(np.r_[0, sites, LENGTH], np.r_[0, shape, 0], lw=1.0, color="0.5")
    axis.plot(sites, shape, "o", ms=6)
    axis.axhline(0, color="0.9", lw=0.8)
    axis.set_title(f"p = {p + 1}\n{modes.frequencies[p]:.3f} rad/s", fontsize=9)
    axis.set_xticks([])
plt.tight_layout()
plt.show()

print(f"solver     : {np.round(modes.frequencies, 6)}")
print(f"closed form: {np.round(exact, 6)}")
print(f"agreement  : {np.max(np.abs(modes.frequencies - exact)) / exact.max():.2e} relative")
print()
print(f"the fastest mode is {modes.frequencies[-1]:.4f} rad/s against the ceiling {CEILING}")
print("a 100-mass chain of the same parts tops out at "
      f"{coupled.chain_mode_frequencies(100, MASS, STIFFNESS)[-1]:.4f} rad/s — "
      "twenty times the masses, the same ceiling")

## חלק 4 — הניסוי המפריך: פרטו וצפו באופנים

משכו שרשרת בת עשרים מסות למשולש ושחררו. שתי תמונות של אותה תנועה, מאותו מסלול: האנרגיה השייכת
לכל מסה, והאנרגיה השייכת לכל אופן.

המסלול מגיע מ-`simulate_coupled`, שמקדם עשרים משוואות מצומדות ומעולם לא שמע על אופן. מה שלא
יעשו אנרגיות האופנים, איש לא אמר להן לעשות זאת.

In [ ]:
N_CHAIN = 20
chain = coupled.chain_matrices(N_CHAIN, MASS, STIFFNESS)
chain_modes = coupled.normal_mode_solve(*chain)

site = np.arange(1, N_CHAIN + 1)
pluck = 0.05 * np.minimum(site, N_CHAIN + 1 - site) / ((N_CHAIN + 1) / 2)

slowest = 2 * np.pi / chain_modes.frequencies[0]
dt = (2 * np.pi / chain_modes.frequencies[-1]) / 200
run = coupled.simulate_coupled(*chain, pluck, np.zeros(N_CHAIN), dt, int(3 * slowest / dt))

site_energy = coupled.site_energies(*chain, run.positions, run.velocities)
modal_energy = coupled.modal_energies(chain_modes, run.positions, run.velocities)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
for mass in (1, 5, 10):
    left.plot(run.times, site_energy[:, mass - 1], lw=1.0, label=f"mass {mass}")
for mode in (1, 2, 3):
    right.plot(run.times, modal_energy[:, mode - 1], lw=1.2, label=f"mode {mode}")
left.set_title("energy per mass", fontsize=10)
right.set_title("energy per mode", fontsize=10)
for axis in (left, right):
    axis.set_xlabel("t (s)"), axis.set_ylabel("energy (J)"), axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

swing = np.ptp(site_energy / site_energy.sum(axis=1, keepdims=True), axis=0)
total = modal_energy[0].sum()
modal_drift = np.max(np.abs(modal_energy - modal_energy[0])) / total

# A symmetric pluck is orthogonal to every even mode, so those hold *exactly* zero energy —
# and a relative drift on a mode holding 1e-31 of the total is a meaningless enormous number.
# Quote the per-mode figure only for the modes that actually carry something.
carrying = modal_energy[0] > 1e-3 * total
per_mode = np.max(np.abs(modal_energy[:, carrying] - modal_energy[0, carrying])
                  / modal_energy[0, carrying])

print(f"site energies: the widest share swings by {swing.max():.1%} of the total")
print(f"modal energies: constant to {modal_drift:.1e} of the total, and to {per_mode:.1e}")
print(f"                of their own value for the {carrying.sum()} modes that hold any")
print(f"total energy drifts by {np.ptp(site_energy.sum(axis=1)) / site_energy.sum(axis=1)[0]:.1e}")
print()
print(f"The other {N_CHAIN - carrying.sum()} modes hold exactly zero — a symmetric pluck is")
print("orthogonal to every even mode — and zero stays zero. Ask a relative question about")
print("those and you get nonsense, which is worth knowing before you write the assertion.")
print()
print("The residual is the integrator's, not the physics'. Halve dt above and it falls")
print("fourfold — velocity Verlet is second order — while the site swing does not budge.")

## חלק 5 — מדוע הצליל נקי

אנרגיות האופנים קבועות, ולכן הספקטרום של השרשרת נקבע ברגע שאתם משחררים. הביטו כיצד הוא מתחלק:
זו התשובה לחידה של המודול.

אחר כך הביטו היכן אתם פורטים. גיטריסט שפורט ליד הגשר מקבל צליל בהיר יותר, והסיבה היא אינטגרל
הטלה.

In [ ]:
def modal_spectrum(shape):
    """Fraction of the total energy each mode of the 20-chain holds, for a starting shape."""
    energy = coupled.modal_energies(chain_modes, shape, np.zeros(N_CHAIN))
    return energy / energy.sum()

quarter = N_CHAIN // 4
off_centre = 0.05 * np.where(
    site <= quarter, site / quarter, (N_CHAIN + 1 - site) / (N_CHAIN + 1 - quarter)
)

fig, (shapes_axis, bars_axis) = plt.subplots(1, 2, figsize=(11, 3.2))
shapes_axis.plot(np.r_[0, site, N_CHAIN + 1], np.r_[0, pluck, 0], "o-", ms=4, label="centre")
shapes_axis.plot(np.r_[0, site, N_CHAIN + 1], np.r_[0, off_centre, 0], "s-", ms=4, label="quarter")
shapes_axis.set_xlabel("mass"), shapes_axis.set_ylabel("initial displacement (m)")
shapes_axis.legend(fontsize=8)

width = 0.4
bars_axis.bar(site - width / 2, modal_spectrum(pluck), width, label="centre")
bars_axis.bar(site + width / 2, modal_spectrum(off_centre), width, label="quarter")
bars_axis.set_xlabel("mode p"), bars_axis.set_ylabel("share of the energy")
bars_axis.set_xticks([1, 5, 10, 15, 20]), bars_axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

for name, shape in (("centre", pluck), ("quarter", off_centre)):
    share = modal_spectrum(shape)
    print(f"{name:8s} pluck: mode 1 holds {share[0]:.1%}, modes 1-5 hold {share[:5].sum():.1%}")

q0, _ = coupled.mode_coordinates(chain_modes, pluck, np.zeros(N_CHAIN))
print()
print(f"centre pluck, modal coordinates q_1..q_6: {np.round(q0[:6], 6)}")
print(f"q1/q3 = {q0[0] / q0[2]:+.3f} and q1/q5 = {q0[0] / q0[4]:+.3f}")
print("Those are the triangle wave's 1/p^2 Fourier coefficients from module 03, with the")
print("alternating sign, and the even modes vanish because the shape is symmetric and they")
print("are not. The departures from exactly -9 and +25 are the cost of sampling a triangle")
print("at 20 points rather than continuously.")

## חלק 6 — ניסוי ההתכנסות

עכשיו המדידה שסביבה בנוי המודול. החזיקו את אורך השרשרת קבוע, הוסיפו מסות, וצפו בתדירויות
הנמוכות מפסיקות לזוז.

פרמטר העידון הוא מספר ה*קפיצים*, $N+1$, מפני שזו הרשת: $N+1$ קפיצים פורשים את האורך. שימוש
ב-$N$ במקום מדווח סדר של 1.95 ומאשים את הפיזיקה בציר שסומן לא נכון — סעיף (ב) של בעיה 6 מבקש
מכם לומר מדוע.

In [ ]:
SIZES = [10, 20, 40, 80, 160]


def frequency_ratio(springs, p):
    """Mode p of a chain with `springs` springs, over the string frequency it tends to."""
    n = springs - 1
    return float(
        coupled.chain_mode_frequencies(n, MASS, STIFFNESS)[p - 1]
        / coupled.chain_continuum_frequencies(n, MASS, STIFFNESS)[p - 1]
    )


fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
for p in (1, 2, 3):
    study = convergence_study(lambda s, p=p: frequency_ratio(s, p), [n + 1 for n in SIZES], 1.0)
    label = f"p = {p} ({study.observed_order:.3f})"
    left.loglog(study.refinements, study.errors, "o-", ms=4, label=label)
    left.loglog(study.refinements, (p * np.pi) ** 2 / (24 * study.refinements ** 2),
                lw=0.9, ls="--", color="0.6")
    print(f"mode {p}: order {study.observed_order:.4f}; "
          f"errors {np.array2string(study.errors, precision=2)}")
left.set_xlabel("N + 1 (springs)"), left.set_ylabel("relative error")
left.legend(fontsize=8, title="mode (fitted order)", title_fontsize=8)

# The error also grows with p, which is the half that limits how far the continuum reaches.
n = 100
chain_f = coupled.chain_mode_frequencies(n, MASS, STIFFNESS)
string_f = coupled.chain_continuum_frequencies(n, MASS, STIFFNESS)
right.loglog(np.arange(1, n + 1), np.abs(chain_f - string_f) / string_f, lw=1.4)
right.axhline(0.01, color="crimson", lw=0.9, ls="--")
right.set_xlabel("mode p, at N = 100"), right.set_ylabel("relative error")
plt.tight_layout()
plt.show()

within = np.where(np.abs(chain_f - string_f) / string_f < 0.01)[0]
print()
print(f"at N = 100, mode 1 is right to {abs(chain_f[0] - string_f[0]) / string_f[0]:.1e}")
print(f"            modes 1..{within[-1] + 1} are within 1% — and there are 100 of them")
top_error = abs(chain_f[-1] - string_f[-1]) / string_f[-1]
print(f"            the top mode is {top_error:.0%} off, for ever")

## חלק 7 — יחס הנפיצה, ועקומה אחת לכל שרשרת

אופניה של כל שרשרת משורטטים כנקודות $(k_p a,\ \omega_p)$ מול העקומה
$\omega(k) = 2\sqrt{k_s/m}\,|\sin(ka/2)|$. שום דבר אינו מותאם. שימו לב מה משותף לשלוש
השרשראות ומה לא.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))

grid = np.linspace(0, np.pi, 400)
left.plot(grid, CEILING * np.abs(np.sin(grid / 2)), lw=1.2, color="0.5", label="omega(k)")
for n, marker in ((5, "o"), (20, "s"), (100, ".")):
    spacing = LENGTH / (n + 1)
    wavenumbers = np.arange(1, n + 1) * np.pi / LENGTH
    left.plot(wavenumbers * spacing, coupled.chain_mode_frequencies(n, MASS, STIFFNESS),
              marker, ms=5 if n < 100 else 3, ls="none", label=f"N = {n}")
    curve = coupled.chain_dispersion(wavenumbers, spacing, MASS, STIFFNESS)
    print(f"N = {n:3d}: modes lie on the curve to "
          f"{np.max(np.abs(coupled.chain_mode_frequencies(n, MASS, STIFFNESS) - curve)):.1e} rad/s")
left.set_xlabel("k a (rad)"), left.set_ylabel("omega (rad/s)"), left.legend(fontsize=8)
left.axvline(np.pi, color="0.85", lw=0.8, ls=":")

# The small-k end, where the curve is a straight line and the chain behaves like a string.
n = 100
spacing = LENGTH / (n + 1)
speed = spacing * np.sqrt(STIFFNESS / MASS)
wavenumbers = np.arange(1, n + 1) * np.pi / LENGTH
low = wavenumbers * spacing < 0.6
right.plot(wavenumbers[low] * spacing, coupled.chain_mode_frequencies(n, MASS, STIFFNESS)[low],
           "o", ms=4, label="N = 100")
right.plot(wavenumbers[low] * spacing, speed * wavenumbers[low], lw=1.0, ls="--",
           color="crimson", label="c k")
right.set_xlabel("k a (rad)"), right.set_ylabel("omega (rad/s)"), right.legend(fontsize=8)
plt.tight_layout()
plt.show()

measured = coupled.chain_mode_frequencies(n, MASS, STIFFNESS)[0] / wavenumbers[0]
print()
print(f"wave speed c = a sqrt(k_s/m) = {speed:.6f} m/s")
print(f"measured from the lowest mode = {measured:.6f} m/s"
      f"  ({100 * (speed - measured) / speed:.4f}% low)")
print(f"predicted shortfall (pi/(N+1))^2/24 = {100 * (np.pi / (n + 1)) ** 2 / 24:.4f}%")

## חלק 8 — מדידה, עם סרגל שגיאה

חמש מסות, אחת מהן נמשכת הצידה ומשוחררת, ורישום רועש של *אותה מסה עצמה* — וזה כל מה שניסוי
אמיתי מקבל. קראו את חמש תדירויות האופנים מתוך הספקטרום שלו.

מודול 06 הסתיים בשתי דרכים למספר אחד שחלקו זו על זו הרבה מעבר לסרגלי השגיאה שלהן. כאן יש מלכודת
אחרת. הריצו את התא, ואז הביטו היטב במה שקורה לתשובה כשהרעש עולה פי חמישה-עשר.

In [ ]:
N_MEAS = 5
meas = coupled.chain_matrices(N_MEAS, MASS, STIFFNESS)
meas_modes = coupled.normal_mode_solve(*meas)
truth = coupled.chain_mode_frequencies(N_MEAS, MASS, STIFFNESS)

DT = 0.02
N_SAMPLES = 8192
# `spectrum` reads a record centred on t = 0, and a chain released from rest moves evenly in
# time — so evaluating `evolve` on the centred grid gives a genuinely even record, with no
# mirroring trick anywhere. This is what the exact modal solution buys you.
times = (np.arange(N_SAMPLES) - N_SAMPLES // 2) * DT
clean = coupled.evolve(meas_modes, np.array([1.0, 0, 0, 0, 0]), np.zeros(N_MEAS), times)[:, 0]


def read_modes(record):
    """The five strongest spectral peaks, located to sub-bin precision, in frequency order."""
    omega, transform = fourier.spectrum(record, DT)
    band = (omega > 0) & (omega < 1.3 * truth[-1])
    magnitude, frequencies = np.abs(transform[band]), omega[band]
    peaks = [i for i in range(1, magnitude.size - 1)
             if magnitude[i] > magnitude[i - 1] and magnitude[i] > magnitude[i + 1]]
    peaks.sort(key=lambda i: magnitude[i], reverse=True)
    vertex = [frequencies[i] + 0.5 * (magnitude[i - 1] - magnitude[i + 1])
              / (magnitude[i - 1] - 2 * magnitude[i] + magnitude[i + 1])
              * (frequencies[1] - frequencies[0]) for i in peaks[:N_MEAS]]
    return np.array(sorted(vertex))


omega_axis, spectrum = fourier.spectrum(measurement.add_noise(clean, 0.10, rng), DT)
show = (omega_axis > 0) & (omega_axis < 1.3 * truth[-1])
fig, axis = plt.subplots(figsize=(9, 3))
axis.semilogy(omega_axis[show], np.abs(spectrum[show]), lw=0.8)
for f in truth:
    axis.axvline(f, color="crimson", lw=0.8, ls="--")
axis.set_xlabel("omega (rad/s)"), axis.set_ylabel("|X|")
plt.tight_layout()
plt.show()

print(f"{'noise':>6}  " + "  ".join(f"omega_{p + 1:d}" for p in range(N_MEAS)) + "   worst error")
for sigma in (0.02, 0.10, 0.30):
    def measure(generator, p=0, s=sigma):
        return float(read_modes(measurement.add_noise(clean, s, generator))[p])

    studies = [seed_study(lambda r, p=p: measure(r, p), n_seeds=8, base_seed=11)
               for p in range(N_MEAS)]
    means = np.array([s.mean for s in studies])
    errors = np.array([s.standard_error for s in studies])
    print(f"{sigma:6.2f}  " + "  ".join(f"{m:7.4f}" for m in means)
          + f"   {100 * np.max(np.abs(means - truth) / truth):.3f}%")
    print(f"{'+/-':>6}  " + "  ".join(f"{e:7.4f}" for e in errors))
print(f"{'exact':>6}  " + "  ".join(f"{f:7.4f}" for f in truth))

In [ ]:
# The error did not move with the noise. Change the one thing that does move it.
print("record   worst error")
for samples in (1024, 2048, 4096, 8192, 16384):
    grid = (np.arange(samples) - samples // 2) * DT
    record = coupled.evolve(meas_modes, np.array([1.0, 0, 0, 0, 0]), np.zeros(N_MEAS), grid)[:, 0]

    omega, transform = fourier.spectrum(record, DT)
    band = (omega > 0) & (omega < 1.3 * truth[-1])
    magnitude, frequencies = np.abs(transform[band]), omega[band]
    peaks = [i for i in range(1, magnitude.size - 1)
             if magnitude[i] > magnitude[i - 1] and magnitude[i] > magnitude[i + 1]]
    peaks.sort(key=lambda i: magnitude[i], reverse=True)
    found = np.array(sorted(
        frequencies[i] + 0.5 * (magnitude[i - 1] - magnitude[i + 1])
        / (magnitude[i - 1] - 2 * magnitude[i] + magnitude[i + 1])
        * (frequencies[1] - frequencies[0]) for i in peaks[:N_MEAS]))
    print(f"{samples * DT:6.1f}s  {100 * np.max(np.abs(found - truth) / truth):.4f}%")

print()
print("These records carry no noise at all, and the error is the same 0.2% the noisy runs")
print("reported. It is a systematic: a finite record has a finite resolution, and the")
print("three-point vertex that interpolates between bins is exact only for a parabola.")
print("Doubling the record halves it. Averaging more seeds does nothing to it whatever.")
print()
print("So the +/- printed above is honest and almost irrelevant. Quoting it alone would")
print("claim four significant figures on a number good to three. Module 06's trap was a")
print("measurement that fell apart under noise; this one is a measurement whose error bar")
print("is not measuring its error.")

In [ ]:
# 1. The general machinery reproduces module 06's hand-solved pair.
assert np.max(np.abs(pair_modes.frequencies - by_hand)) < 1e-12
assert np.allclose(pair_modes.shapes.T @ pair[0] @ pair_modes.shapes, np.eye(2), atol=1e-12)

# 2. The chain's closed form and the general eigensolver agree, at every size.
for n in (1, 5, 20, 100):
    solved = coupled.normal_mode_solve(*coupled.chain_matrices(n, MASS, STIFFNESS))
    assert np.allclose(solved.frequencies, coupled.chain_mode_frequencies(n, MASS, STIFFNESS),
                       rtol=1e-12)
    assert solved.frequencies[-1] < CEILING  # the band top is never reached

# 3. The falsifier: modal energies constant while site energies slosh.
assert modal_drift < 1e-3
assert swing.max() > 0.1

# 4. A plucked chain is mostly its fundamental — the answer to the puzzle.
assert modal_spectrum(pluck)[0] > 0.8

# 5. The continuum limit converges at second order in the number of springs.
order = convergence_study(lambda s: frequency_ratio(s, 1), [n + 1 for n in SIZES], 1.0)
assert abs(order.observed_order - 2.0) < 0.05

# 6. Every mode of every chain lies on one dispersion curve.
for n in (5, 20, 100):
    spacing_n = LENGTH / (n + 1)
    k_n = np.arange(1, n + 1) * np.pi / LENGTH
    assert np.allclose(coupled.chain_mode_frequencies(n, MASS, STIFFNESS),
                       coupled.chain_dispersion(k_n, spacing_n, MASS, STIFFNESS), atol=1e-12)

# 7. The noisy measurement finds all five frequencies, and its error is the record's.
recovered = read_modes(measurement.add_noise(clean, 0.10, np.random.default_rng(1)))
assert np.max(np.abs(recovered - truth) / truth) < 0.005

print("all checks passed")
print(f"{N_CHAIN} modes held their energy to {modal_drift:.1e} while the masses traded theirs")

## חלק 9 — לחקור בעצמכם

שנו את $N$ ואת תנאי השפה וצפו במה שקורה לספקטרום. שלושה דברים שכדאי לחפש.

- העלו את $N$ עם קצוות קבועים וצפו בתדירות הנמוכה ביותר יורדת בעוד הגבוהה ביותר נעצרת.
- עברו ל-`free` ומצאו את אופן האפס. זה אינו באג; זהו שימור התנע.
- עברו ל-`periodic` וחפשו זוגות של תדירויות שוות. אחר כך שאלו איזו סימטריה יש לטבעת שאין
  לשרשרת בין שני קירות.

In [ ]:
import ipywidgets as widgets


def explore(n=8, boundary="fixed"):
    built = coupled.chain_matrices(n, MASS, STIFFNESS, boundary)
    solved = coupled.normal_mode_solve(*built)
    spacing = LENGTH / (n + 1)

    fig, (shapes_axis, spectrum_axis) = plt.subplots(1, 2, figsize=(10.5, 3.2))
    places = np.arange(1, n + 1) * spacing
    for p in range(min(3, n)):
        shape = solved.shapes[:, p] / np.abs(solved.shapes[:, p]).max()
        shapes_axis.plot(places, shape, "o-", ms=4, lw=1.0, label=f"p = {p + 1}")
    shapes_axis.axhline(0, color="0.9", lw=0.8)
    shapes_axis.set_xlabel("position (m)"), shapes_axis.set_ylabel("shape (scaled)")
    shapes_axis.legend(fontsize=8)

    spectrum_axis.plot(np.arange(1, n + 1), solved.frequencies, "o", ms=5)
    spectrum_axis.axhline(CEILING, color="crimson", lw=0.9, ls="--")
    spectrum_axis.set_ylim(-0.3, 1.1 * CEILING)
    spectrum_axis.set_xlabel("mode p"), spectrum_axis.set_ylabel("omega (rad/s)")
    plt.tight_layout()
    plt.show()

    zero = solved.frequencies < 1e-6 * solved.frequencies[-1]
    gaps = np.diff(solved.frequencies)
    print(f"{n} modes from {solved.frequencies[0]:.4f} to {solved.frequencies[-1]:.4f} rad/s")
    print(f"zero modes: {zero.sum()}   degenerate pairs: {int(np.sum(gaps < 1e-9))}")


widgets.interact_manual(
    explore,
    n=widgets.IntSlider(min=2, max=40, step=1, value=8, description="N"),
    boundary=widgets.Dropdown(options=["fixed", "free", "periodic"], description="ends"),
);

## בדקו את הבנתכם

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "07-normal-modes.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלמטה.

1. בחלק 4 אנרגיות האופנים היו שטוחות ואנרגיות האתרים לא, מתוך אחד ואותו מסלול. הסבירו למי שעשה
   רק את מודול 06 מדוע אין כאן סתירה, ומה הוא היה אמור לצפות לו בהינתן מה שעשו אותן מטוטלות.
2. בחלק 6 התאמתם סדר התכנסות בעזרת $N+1$ כפרמטר העידון וקיבלתם 2.000. שימוש ב-$N$ נותן 1.95.
   שניהם אריתמטיקה ישרה על אותם מספרים. אמרו איזה מהם הייתם מדווחים וכיצד הייתם מצדיקים את
   הבחירה בפני מי שחושב שבחרתם את זה שקרוב יותר ל-2.
3. סרגל השגיאה של חלק 8 הצטמצם לאין בעוד השגיאה נשארה במקומה. אמרו מה הייתם מדווחים כאי-ודאות
   על $\omega_1$, ומה הייתם צריכים לעשות למערך הניסוי — לא לניתוח — כדי להקטין אותה.

**התשובות שלכם:**

1.
2.
3.